# Multi-Asset Portfolio Risk Analysis

## Research Question

How does diversification across asset classes affect the historical
risk and return characteristics of a portfolio?

### Secondary Question

How do differences in the correlations between asset returns
contribute to portfolio diversification?

## Asset Selection

This analysis uses five exchange-traded funds (ETFs) representing
different market exposures:

| Ticker | Exposure |
|---|---|
| SPY | U.S. large-cap equities |
| QQQ | Nasdaq-100 equities |
| VXUS | International equities excluding the U.S. |
| IEF | U.S. Treasury bonds |
| GLD | Gold |

The assets were selected to provide a mixture of equity, fixed-income,
international, and alternative-asset exposures for examining historical
portfolio diversification.

## Analysis Period

The analysis covers January 1, 2020 through December 31, 2025 using
historical daily market data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

In [3]:
tickers = ["SPY", "QQQ", "VXUS", "IEF", "GLD"]

start_date = "2020-01-01"
end_date = "2026-01-01"

In [5]:
raw_data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False
)

In [6]:
raw_data.head()

Price        Adj Close                                                \
Ticker             GLD        IEF         QQQ         SPY       VXUS   
Date                                                                   
2020-01-02  143.949997  93.721725  207.872253  295.391754  46.430737   
2020-01-03  145.860001  94.348091  205.968140  293.154968  45.861805   
2020-01-06  147.389999  94.246498  207.295242  294.273346  45.927769   
2020-01-07  147.970001  94.111115  207.266434  293.445923  45.845318   
2020-01-08  146.860001  93.890999  208.824265  295.009949  45.927769   

Price            Close                                                 ...  \
Ticker             GLD         IEF         QQQ         SPY       VXUS  ...   
Date                                                                   ...   
2020-01-02  143.949997  110.730003  216.160004  324.869995  56.310001  ...   
2020-01-03  145.860001  111.470001  214.179993  322.410004  55.619999  ...   
2020-01-06  147.389999  111.349998  215.559998  323.640015  55.700001  ...   
2020-01-07  147.970001  111.190002  215.529999  322.730011  55.599998  ...   
2020-01-08  146.860001  110.930000  217.149994  324.450012  55.700001  ...   

Price             Open                                                 \
Ticker             GLD         IEF         QQQ         SPY       VXUS   
Date                                                                    
2020-01-02  143.860001  110.690002  214.399994  323.540009  56.169998   
2020-01-03  145.750000  111.150002  213.300003  321.160004  55.590000   
2020-01-06  148.440002  111.660004  212.500000  320.489990  55.419998   
2020-01-07  147.570007  111.330002  215.639999  323.019989  55.740002   
2020-01-08  148.490005  111.290001  215.500000  322.940002  55.590000   

Price         Volume                                        
Ticker           GLD      IEF       QQQ       SPY     VXUS  
Date                                                        
2020-01-02   7733800  4022300  30969400  59151200  2017900  
2020-01-03  12272800  3839600  27518900  77709700  2140200  
2020-01-06  14403300  2714300  21655300  55653900  2119900  
2020-01-07   7978500  2038800  22139300  40496400  1987300  
2020-01-08  22248500  5081100  26397300  68296000  1824400  

[5 rows x 30 columns]

In [7]:
raw_data.shape

(1508, 30)

In [8]:
raw_data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 1508 entries, 2020-01-02 to 2025-12-31
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   (Adj Close, GLD)   1508 non-null   float64
 1   (Adj Close, IEF)   1508 non-null   float64
 2   (Adj Close, QQQ)   1508 non-null   float64
 3   (Adj Close, SPY)   1508 non-null   float64
 4   (Adj Close, VXUS)  1508 non-null   float64
 5   (Close, GLD)       1508 non-null   float64
 6   (Close, IEF)       1508 non-null   float64
 7   (Close, QQQ)       1508 non-null   float64
 8   (Close, SPY)       1508 non-null   float64
 9   (Close, VXUS)      1508 non-null   float64
 10  (High, GLD)        1508 non-null   float64
 11  (High, IEF)        1508 non-null   float64
 12  (High, QQQ)        1508 non-null   float64
 13  (High, SPY)        1508 non-null   float64
 14  (High, VXUS)       1508 non-null   float64
 15  (Low, GLD)         1508 non-null   float64
 16  (Low, IEF)       

In [9]:
prices = raw_data["Adj Close"].copy()

In [10]:
prices.head()

Ticker,GLD,IEF,QQQ,SPY,VXUS
Date,,,,,
2020-01-02,143.949997,93.721725,207.872253,295.391754,46.430737
2020-01-03,145.860001,94.348091,205.968140,293.154968,45.861805
2020-01-06,147.389999,94.246498,207.295242,294.273346,45.927769
2020-01-07,147.970001,94.111115,207.266434,293.445923,45.845318
2020-01-08,146.860001,93.890999,208.824265,295.009949,45.927769


In [11]:
prices.tail()

Ticker,GLD,IEF,QQQ,SPY,VXUS
Date,,,,,
2025-12-24,411.929993,93.836914,621.812256,685.029480,75.057564
2025-12-26,416.739990,93.924576,621.772400,684.959961,75.295815
2025-12-29,398.600006,94.060913,618.762634,682.519043,75.057564
2025-12-30,398.890015,93.963516,617.327515,681.685608,75.206474
2025-12-31,396.309998,93.651878,612.224915,676.635010,74.888809


In [12]:
prices.isna().sum()

Ticker
GLD     0
IEF     0
QQQ     0
SPY     0
VXUS    0
dtype: int64

In [13]:
prices.to_csv("../data/adjusted_close_prices.csv")

In [14]:
prices.plot(figsize=(12, 6))

plt.title("Historical Adjusted Prices")
plt.xlabel("Date")
plt.ylabel("Adjusted Price (USD)")
plt.tight_layout()
plt.show()

<Figure size 1200x600 with 1 Axes>